# Train Self-Attention Generative Adversarial Network (SAGAN) <BR>on MNIST Dataset

Source:

https://github.com/franknb/Self-attention-DCGAN/blob/master/SAGAN_mnist.ipynb

Adapted by:

Antonio Esteves @ UMinho, Jul 2024

---
TODO:

* Modify `'OUR_WANDB_PROJECT_ID'`
* Modify `'OUR_WANDB_ENTITY'`
* Modify `config["experiment_name"]`
* Modify `'OUR_DATASET_PATH'`
---

## Import the Necessary Libraries

In [ ]:
import datetime
import time
import os
import wandb
import cv2
import numpy                as     np
from   IPython.display      import clear_output
import matplotlib.pyplot    as     plt

import torch
import torch.nn             as     nn
import torch.nn.functional  as     F
from   torch.utils.data     import DataLoader
from   torchvision          import transforms
from   torchvision.utils    import save_image
from   torchvision.datasets import MNIST
from   torch.nn             import Parameter

## Configuration

In [ ]:
# Setup device agnostic code

device = "cuda" if torch.cuda.is_available() else "cpu"

print(f'Using {device} for computing')

# Hyperparameters
config = {}

config["batch_size"]        = 64
config["log_interval"]      = 100
config["sampling_interval"] = 1000
config["checkp_interval"]   = 10000
config["experiment_name"]   = "SAGAN_MNIST_02"

# Location where we will save here the images generated during SAGAN training
RESULTS_PATH = f'results/{config["experiment_name"]}'
os.makedirs(RESULTS_PATH, exist_ok=True)

## Login into Weights and Bias

In [ ]:
wandb.login()

## Track metadata and hyperparameters with Weights and Bias

Define the experiment: the hyperparameters, the dataset and model name. This information will be stored in a `config` dictionary.

In [ ]:
config_wandb = config

wandb.init(
    project = 'OUR_WANDB_PROJECT_ID',
    entity  = 'OUR_WANDB_ENTITY', 
    config  = config_wandb
)

## Setup the Training DataLoader

In [ ]:
def denorm(x):
    out = (x + 1) / 2
    return out.clamp_(0, 1)

# Define data transformer
img_transform = transforms.Compose([
    transforms.Resize(28),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

# Read data and transform
dataset    = MNIST(
    root      = 'OUR_DATASET_PATH',
    download  = False,
    train     = True,
    transform = img_transform,
)

dataloader = DataLoader(
    dataset,
    batch_size = config["batch_size"],
    shuffle    = True,
)

# Create a fixed latent vector containing random numbers
fixed_z = torch.randn(64, 100).to(device)

## Self-attention Layer

1. Uses convolutions with a kernel of size 1 to project the input features into the $f$ (Query), $g$ (Key) and $h$ (Value) spaces, using a shape equal to $(Channels \times N)$, where $N = Width * Height$.

2. Generates the attention score $S$ through a matrix dot product between transposed $Query$ and $Key$, with the shape of $(N \times N)$. The $N \times N$ attention scores $S$ describes the attention that each pixel puts on every other pixel, hence the name 'self-attention'.
3. The attention score $S$ is normalized using $softmax$.
4. The self-attention map is obtained with dot product between the normalized attention scores and the $Value$. The output has a shape equal $(C \times N)$.
5. The obtained self-attention map is reshaped to have the same shape as the input features $x$, i.e., $(C \times H \times W)$. The output is $o$.
6. Self-attention map $o$ is multiplied by the learnable parameter $\gamma$ and then added to the input features $x$, to produce the modified self-attention feature map $y$. The shape of $y$ is equal to the shape of the input features $x$, i.e., $(C \times H \times W)$. The parameter $\gamma$ is initialized at 0.

![](../fig/SAGAN_attention_layer1.png)



In [ ]:
class Self_Attn(nn.Module):
    """
    Self-attention Layer.
    """
    def __init__(self, in_dim):
        super().__init__()

        # Build the module
        self.query_conv = nn.Conv2d( # 1 X 1 convolution
            in_channels  = in_dim,
            out_channels = in_dim // 2,
            kernel_size  = 1,
        )
        self.key_conv   = nn.Conv2d( # 1 X 1 convolution
            in_channels  = in_dim,
            out_channels = in_dim // 2,
            kernel_size  = 1,
        )
        self.value_conv = nn.Conv2d( # 1 X 1 convolution
            in_channels  = in_dim,
            out_channels = in_dim,
            kernel_size  = 1,
        )
        self.gamma      = nn.Parameter(torch.zeros(1))
        self.softmax    = nn.Softmax(dim = -1)

    def forward(self, x):
        """
            inputs :
                x : input feature maps( BS * C * H * W)
            returns :
                out : self-attention value added to the input feature 'x'
                attention: BS * N * N (N is Width*Height)
        """
        bs, C, height, width = x.size()

        proj_query = self.query_conv(x).view(bs, -1, height*width).permute(0,2,1) # BS x (H*W) x C
        proj_key   = self.key_conv(x).view(bs, -1, height*width)                  # BS x C x (H*W)
        s_scores   = torch.bmm(proj_query, proj_key)                              # batch matrix-matrix product

        attention  = self.softmax(s_scores)                          # BS x (H*W) x (H*W)
        proj_value = self.value_conv(x).view(bs, -1, height*width)   # BS x C x (H*W)
        o          = torch.bmm(proj_value, attention.permute(0,2,1)) # batch matrix-matrix product
        o          = o.view(bs,C,width,height)                       # BS x C x H x W

        y          = self.gamma * o + x

        return y, attention

## Spectral Normalization Layer

[Source](https://paperswithcode.com/method/spectral-normalization)

Spectral Normalization (SN) is a weight normalization technique proposed in [Spectral Normalization for Generative Adversarial Networks](https://arxiv.org/abs/1802.05957) to stabilize the training process.

 ![](../fig/SAGAN_spectral_normalization1.png)

Spectral normalization divides the weights $W_i$ by their spectral norms $\sigma(W_i)$, that is, the largest singular value of $W_i$. In this way, it mitigates two sources of failure during GAN training: exploding and vanishing gradients.

SN controls the Lipschitz constant of the discriminator $D$ by constraining the spectral norm of each of its layers $g: \textbf{h}_{in} \rightarrow \textbf{h}_{out}$. The Lipschitz norm $\Vert{g}\Vert_{\text{Lip}}$ is equal to $\sup_{\textbf{h}}\sigma\left(\nabla{g}\left(\textbf{h}\right)\right)$, where $\sigma\left(a\right)$ is the spectral norm of the matrix 
$A$ ($L_2$ matrix norm of $A$):

$$ \sigma \left( a \right)
    = \max_{\textbf{h}:\textbf{h}\neq{0}}\frac{\Vert{A\textbf{h}}\Vert_{2}}{\Vert\textbf{h}\Vert_{2}} 
    = \max_{\Vert\textbf{h}\Vert_{2}\leq{1}}{\Vert{A\textbf{h}}\Vert_{2}}$$

which is equivalent to the largest singular value of $A$. Therefore for a linear layer $g\left(\textbf{h}\right) = W\textbf{h}$ the norm is given by $\Vert{g}\Vert_{\text{Lip}} = \sup_{\textbf{h}}\sigma\left(\nabla{g}\left(\textbf{h}\right)\right) = \sup_{\textbf{h}}\sigma\left(W\right) = \sigma\left(W\right)$.

Spectral normalization normalizes the spectral norm of the weight matrix $W$ so it satisfies the Lipschitz constraint $\sigma\left(W\right) = 1$:

 $\bar{W}_{\text{SN}}\left(W\right) = \frac{W}{\sigma\left(W\right)}$


In [ ]:
def l2normalize(v, eps=1e-12):
    return v / (v.norm() + eps)


class SpectralNorm(nn.Module):
    '''
    Spectral Normalization layer.
    Source:
       https://github.com/heykeetae/Self-Attention-GAN/blob/master/spectral.py
    '''
    def __init__(self, module, name='weight', power_iterations=1):
        super(SpectralNorm, self).__init__()
        self.module = module
        self.name   = name
        self.power_iterations = power_iterations
        if not self._made_params():
            self._make_params()

    def _update_u_v(self):
        u = getattr(self.module, self.name + "_u")
        v = getattr(self.module, self.name + "_v")
        w = getattr(self.module, self.name + "_bar")

        height = w.data.shape[0]
        for _ in range(self.power_iterations):
            v.data = l2normalize(torch.mv(torch.t(w.view(height,-1).data), u.data))
            u.data = l2normalize(torch.mv(w.view(height,-1).data, v.data))

        # sigma = torch.dot(u.data, torch.mv(w.view(height,-1).data, v.data))
        sigma = u.dot(w.view(height, -1).mv(v))
        setattr(self.module, self.name, w / sigma.expand_as(w))

    def _made_params(self):
        try:
            u = getattr(self.module, self.name + "_u")
            v = getattr(self.module, self.name + "_v")
            w = getattr(self.module, self.name + "_bar")
            return True
        except AttributeError:
            return False


    def _make_params(self):
        w      = getattr(self.module, self.name)

        height = w.data.shape[0]
        width  = w.view(height, -1).data.shape[1]

        u      = Parameter(w.data.new(height).normal_(0, 1), requires_grad=False)
        v      = Parameter(w.data.new(width).normal_(0, 1), requires_grad=False)
        u.data = l2normalize(u.data)
        v.data = l2normalize(v.data)
        w_bar  = Parameter(w.data)

        del self.module._parameters[self.name]

        self.module.register_parameter(self.name + "_u", u)
        self.module.register_parameter(self.name + "_v", v)
        self.module.register_parameter(self.name + "_bar", w_bar)


    def forward(self, *args):
        self._update_u_v()
        return self.module.forward(*args)

## Architecture of the Discriminator and Generator

The architecture of the Discriminator and Generator is shown in the figure below.

![](../fig/SAGAN_disc_gen_architectures.png)

### Generator Model

In [ ]:
class Generator(nn.Module):
    """
    Generator model.

    input: 
        z: latent matrix with shape of (batch_size, 100)
    output: 
        out: generated image with shape (batch_size, 1, 28, 28)
        p1: attention matrix generated by attn layer
    """
    def __init__(self, batch_size=64, attn=True, image_size=28, z_dim=100, conv_dim=64):
        super().__init__()
        self.attn_arg = attn
        
        # Layer 1 turn 100 dims -> 512 dims, size 1 -> 3
        layer1 = []
        layer1.append(
            SpectralNorm(
                nn.ConvTranspose2d(
                    in_channels  = z_dim,
                    out_channels = conv_dim*8,
                    kernel_size  = 3,
                )
            )
        )
        layer1.append(nn.BatchNorm2d(conv_dim*8))
        layer1.append(nn.ReLU())
        self.l1 = nn.Sequential(*layer1)
        
        # Layer 2 turn 512 dims -> 256 dims, size 3 -> 7
        layer2 = []
        layer2.append(
            SpectralNorm(
                nn.ConvTranspose2d(
                    in_channels  = conv_dim*8,
                    out_channels = conv_dim*4, 
                    kernel_size  = 3,
                    stride       = 2,
                    padding      = 0,
                )
            )
        )
        layer2.append(nn.BatchNorm2d(conv_dim*4))
        layer2.append(nn.ReLU())
        self.l2 = nn.Sequential(*layer2)
        
        # Layer 3 turn 256 dims -> 128 dims, size 7 -> 14
        layer3 = []
        layer3.append(
            SpectralNorm(
                nn.ConvTranspose2d(
                    in_channels  = conv_dim*4,
                    out_channels = conv_dim*2, 
                    kernel_size  = 4,
                    stride       = 2,
                    padding      = 1,
                )
            )
        )
        layer3.append(nn.BatchNorm2d(conv_dim*2))
        layer3.append(nn.ReLU())
        self.l3 = nn.Sequential(*layer3)

        # Layer 4 (Attn) turn 128 dims -> 128 dims
        self.attn = Self_Attn(conv_dim*2)
        
        # Layer 5 turn 128 dims -> 1 dims, size 14 -> 28
        last = []
        last.append(nn.ConvTranspose2d(conv_dim*2, 1, 4, 2, 1))
        last.append(nn.Tanh())
        self.last = nn.Sequential(*last)

    def forward(self, z):
        # z is the input random matrix for generator
        z   = z.view(z.size(0), z.size(1), 1, 1)
        out = self.l1(z)
        out = self.l2(out)
        out = self.l3(out)
        if self.attn_arg == True:
            out, _ = self.attn(out)
        out = self.last(out)

        return out

### Discriminator Model

In [ ]:
class Discriminator(nn.Module):
    """
    Discriminator model.

    input:
        x: one batch of data with shape of (batch_size, 1, 28, 28)
    output: 
        out.squeeze: a batch of scalars indicating the predict results
        p1: attention matrix generated by attn layer
    """
    def __init__(self, batch_size=64, attn=True, image_size=28, conv_dim=64):
        super().__init__()
        self.attn_arg = attn
        
        layer1    = []
        layer1.append(SpectralNorm(nn.Conv2d(1, conv_dim, 4, 2, 1)))
        layer1.append(nn.LeakyReLU(0.1))
        curr_dim  = conv_dim
        self.l1   = nn.Sequential(*layer1)
        
        layer2    = []
        layer2.append(SpectralNorm(nn.Conv2d(curr_dim, curr_dim * 2, 4, 2, 1)))
        layer2.append(nn.LeakyReLU(0.1))
        curr_dim  = curr_dim * 2
        self.l2   = nn.Sequential(*layer2)
        
        layer3    = []
        layer3.append(SpectralNorm(nn.Conv2d(curr_dim, curr_dim * 2, 4, 2, 1)))
        layer3.append(nn.LeakyReLU(0.1))
        curr_dim  = curr_dim * 2
        self.l3   = nn.Sequential(*layer3)
        
        self.attn = Self_Attn(curr_dim)
        
        last      = []
        last.append(nn.Conv2d(curr_dim, 1, 4, 2, 1))
        self.last = nn.Sequential(*last)

    def forward(self, x):
        out = self.l1(x)
        out = self.l2(out)
        out = self.l3(out)
        if self.attn_arg == True:
            out, _ = self.attn(out)
        out = self.last(out)

        return out.squeeze()

## Functions to save and load the models to/from file

In [ ]:
def save_model_and_results(
        discriminator,
        generator,
        d_optimizer,
        g_optimizer,
        results,
        epoch,
        hyperparameters,
        file_name
    ):
    results_to_save = {
        'discriminator':   discriminator.state_dict(),
        'generator':       generator.state_dict(),
        'd_optimizer':     d_optimizer.state_dict(),
        'g_optimizer':     g_optimizer.state_dict(),
        'results':         results,
        'epoch':           epoch,
        'hyperparameters': hyperparameters,
    }

    torch.save(
        results_to_save,
        file_name,
    )

In [ ]:
def load_model(discriminator, generator, d_optimizer, g_optimizer, file_name, device):
    '''
    Given instances of the generator and discriminator models, loads from file 'file_name':
    (i)   the weights of both models,
    (ii)  the optimizers state,
    (iii) the results obtained during model training and
    (iv)  the training hyperparameters used to train the models,
    and put the models on 'device'.

    Returns the loaded results and the loaded hyperparameters.
    '''

    results_loaded = torch.load(file_name)

    discriminator.load_state_dict(results_loaded['discriminator'])
    discriminator.to(device)

    generator.load_state_dict(results_loaded['generator'])
    generator.to(device)

    d_optimizer.load_state_dict(results_loaded['d_optimizer'])
    g_optimizer.load_state_dict(results_loaded['g_optimizer'])

    # Returns the saved results and the saved hyperparameters
    return results_loaded['results'], results_loaded['epoch'], results_loaded['hyperparameters']

## Function that Implements the Training Loop

In [ ]:
def train(steps = 100000, batch_size = 64, z_dim = 100, attn = True, results=None):

    # Instantiate the generator and discriminator models ..................

    G = Generator(batch_size, attn).to(device)
    D = Discriminator(batch_size, attn).to(device)
    
    # Initialize the optimizers with filter, lr, and model parameters .....

    g_optimizer = torch.optim.Adam(
        filter(lambda p: p.requires_grad, G.parameters()), 
        0.0001, 
        [0.0,0.9],
    )
    d_optimizer = torch.optim.Adam(
        filter(lambda p: p.requires_grad, D.parameters()), 
        0.0004, 
        [0.0,0.9],
    )
    
    # Iterator to load the data
    Iter = iter(dataloader)
    
    # Start timer
    start_time = time.time()
    
    for step in range(steps):

        # ....................... Train the discriminator .................

        D.train() # Put the discriminator in training mode
        G.train() # Put the generator     in training mode

        try:
            real_images,_ = next(Iter)
        except:
            Iter          = iter(dataloader)
            real_images,_ = next(Iter)

        real_images = real_images.to(device)
        
        # Compute the loss with real images
        d_out_real  = D(real_images)
        d_loss_real = torch.nn.ReLU()(1.0 - d_out_real).mean()
        
        # Compute the loss with fake images
        z           = torch.randn(batch_size, z_dim).to(device)
        fake_images = G(z)
        d_out_fake  = D(fake_images)
        d_loss_fake = torch.nn.ReLU()(1.0 + d_out_fake).mean()
        
        # Compute the Hinge adversarial loss for the discriminator
        d_loss = d_loss_real + d_loss_fake

        # Reset loss gradients, backpropagate the loss gradients, and update the model weights
        d_optimizer.zero_grad()
        g_optimizer.zero_grad()
        d_loss.backward()
        d_optimizer.step()
        
        # ..................... Train the generator .......................

        # Create a batch of random noise vectors
        z           = torch.randn(batch_size, z_dim).to(device)
        fake_images = G(z)
        g_out_fake  = D(fake_images)
        
        # Compute the Hinge adversarial loss for the generator
        g_loss_fake = - g_out_fake.mean()

         # Reset loss gradients, backpropagate the loss gradients, and update the model weights
        d_optimizer.zero_grad()
        g_optimizer.zero_grad()
        g_loss_fake.backward()
        g_optimizer.step()

        # Save the results in a dictionary ......................................
        results["d_loss"].append(d_loss.item())
        results["d_r_loss"].append(d_loss_real.item())
        results["d_f_loss"].append(d_loss_fake.item())
        results["g_loss"].append(g_loss_fake.item())
        results["g_gamma"].append(G.attn.gamma.mean().item())

        # Print out log info ..............................................
        if (step + 1) % config["log_interval"] == 0:

            mean_d_loss      = np.mean(results["d_loss"][-config["log_interval"]:])
            mean_d_r_loss    = np.mean(results["d_r_loss"][-config["log_interval"]:])
            mean_d_f_loss    = np.mean(results["d_f_loss"][-config["log_interval"]:])
            mean_g_loss      = np.mean(results["g_loss"][-config["log_interval"]:])
            g_gamma          = G.attn.gamma.mean().item()

            elapsed = time.time() - start_time
            expect  = elapsed/(step + 1)*(steps-step-1)
            elapsed = str(datetime.timedelta(seconds=elapsed))
            expect  = str(datetime.timedelta(seconds=expect))
            clear_output(wait=True)

            print(f'Elapsed [{elapsed}], Expect [{expect}], step [{step+1}/{steps}],', end=' ')
            print(f'D loss: {mean_d_loss :0>9.6f}', end="  ")
            print(f'G loss: {mean_g_loss :0>9.6f}', end="  ")
            print(f'G gamma: {g_gamma :.4f}')

            try:
                # Log metrics to Weights & Biases .........................
                wandb.log(
                    {
                    "d_loss":   mean_d_loss,
                    "d_r_loss": mean_d_r_loss,
                    "d_f_loss": mean_d_f_loss,
                    "g_loss":   mean_g_loss,
                    "g_gamma":  g_gamma,
                    }
                )
            except Exception as ex:
                print(f'An exception of type {type(ex).__name__} occurred. Arguments:\n{ex.args!r}')

        # Sample images ...................................................

        if (step + 1) % config["sampling_interval"] == 0:
            fake_images = G(fixed_z)

            if attn == True:
                post = f'attn_{str(step+1).zfill(5)}.png'
            else:
                post = f'{str(step+1).zfill(5)}.png'
            f_name = f'results/{config["experiment_name"]}/generated_{post}'
            save_image(
                denorm(fake_images),
                f_name,
            )

        # Save the models .................................................

        if ((step+1) % config["checkp_interval"] == 0) or ((step+1) == steps):
            file_save_model = f'models/checkpoints/{config["experiment_name"]}_{str(step+1).zfill(5)}.pth'
            save_model_and_results(
                D,
                G,
                d_optimizer,
                g_optimizer,
                results,
                0, # do not use epochs is this notebook
                config,
                file_save_model,
            )


## Train the SAGAN Model

In [ ]:
# Create an empty dictionary to store the training results .................
results = {
    'd_loss':              [],
    'd_r_loss':            [],
    'd_f_loss':            [],
    'g_loss':              [],
    'g_gamma':             [],
}

train(steps = 60000, batch_size=config["batch_size"], attn = True, results=results)
print('[INFO] Finished training models with attention.')

train(steps = 60000, batch_size=config["batch_size"], attn = False, results=results)
print('[INFO] Finished training models without attention.')

## Generate a GIF containing a sequence of 8x8 digits at different points of training

In [ ]:
from PIL import Image, ImageDraw, ImageFont

font = ImageFont.truetype("../config/arial.ttf", 18)

def create_image_with_text(img, wh, text):
    width, height = wh
    draw = ImageDraw.Draw(img)
    draw.text((width, height), text, font = font, fill="white")
    return img

frames = []

for i in range(config["sampling_interval"], 20001, config["sampling_interval"]):
    img = Image.open(
        f'results/{config["experiment_name"]}/generated_{str(i).zfill(5)}.png'
    )
    img1 = Image.open(
        f'results/{config["experiment_name"]}/generated_attn_{str(i).zfill(5)}.png'
    )
    width, height = img.size
    expand    = Image.new(img.mode, (width*2 + 10, height + 40), "black")
    expand.paste(img, (0, 0))
    expand.paste(img1, (width + 10, 0))
    epoch     = round(i*64/60000,2)
    new_frame = create_image_with_text(expand,(10,258), "After " + str(epoch) + " epochs")
    new_frame = create_image_with_text(new_frame,(10,238), "Without attention")
    new_frame = create_image_with_text(new_frame,(width + 20,238), "With attention")
    frames.append(new_frame)
    
frames[0].save(
    f'results/{config["experiment_name"]}/{config["experiment_name"]}_comparison1.gif',
    format        = 'GIF',
    append_images = frames[1:],
    save_all      = True,
    duration      = 500,
    loop          = 0,
)

## Plot a file containing a grid of 8x8 digits generated during training 

In [ ]:
file_name  = f'results/{config["experiment_name"]}/generated_01000.png'

img = cv2.imread(file_name)
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

plt.imshow(img)
plt.title(f'{config["experiment_name"]} :: iteration 01000')
plt.show()

print(f'[INFO] type={type(img)}  shape={img.shape}')

## Display a few images generated without attention and the same number of images generated with attention

In [ ]:
# Create a numpy array with a row of images generated without attention

STEP_IN_IMAGE_NUMBER = config["sampling_interval"] # 1000
NUMBER_IMAGES_IN_ROW = 7

for i in range(NUMBER_IMAGES_IN_ROW):

    file_num  = STEP_IN_IMAGE_NUMBER * (i+1)
    file_name = f'results/{config["experiment_name"]}/generated_{str(file_num).zfill(5)}.png'

    if i == 0:
        img         = cv2.imread(file_name)
        row_no_attn = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    else:
        img         = cv2.imread(file_name)
        img         = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        row_no_attn = np.concatenate((row_no_attn, img), axis=1)

# Create a numpy array with a row of 7 images generated with attention

for i in range(NUMBER_IMAGES_IN_ROW):

    file_num  = STEP_IN_IMAGE_NUMBER * (i+1)
    file_name = f'results/{config["experiment_name"]}/generated_attn_{str(file_num).zfill(5)}.png'

    if i == 0:
        img      = cv2.imread(file_name)
        row_attn = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    else:
        img      = cv2.imread(file_name)
        img      = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        row_attn = np.concatenate((row_attn, img), axis=1)

# Plot the row of images generated without attention

plt.figure(figsize=(15, 5))
plt.subplot(2, 1, 1)
plt.axis("off")
plt.title(f'Grids of 8x8 images generated without attention | step={STEP_IN_IMAGE_NUMBER} iterations')
plt.imshow(row_no_attn)

# Plot the row of images generated with attention

plt.subplot(2, 1, 2)
plt.axis("off")
plt.title(f'Grids of 8x8 images generated with attention | step={STEP_IN_IMAGE_NUMBER} iterations')
plt.imshow(row_attn)
plt.savefig(f'results/{config["experiment_name"]}/{config["experiment_name"]}_attn_vs_noattn_step1k.png')
plt.show()


In [ ]:
# Mark the Weights and Bias run as finished
wandb.finish()